# Auto ML Test

## Libreria

In [0]:
# 1. Instalación (ejecutar en terminal o notebook)
!pip install "flaml[automl]"

In [0]:
%restart_python

In [0]:
# 2. Imports
import pandas as pd
import os
import numpy as np
from flaml import AutoML
import matplotlib.pyplot as plt

## Data

## Split DataSet

In [0]:

# Ruta base
ruta_base = "/Workspace/Users/jorgee.lopez@adres.gov.co/mlops_canvas/notebooks/data_prueba"

# Leer archivos parquet
train_df = pd.read_parquet(os.path.join(ruta_base, "train_df.parquet"))
test_df = pd.read_parquet(os.path.join(ruta_base, "test_df.parquet"))
backtest_df = pd.read_parquet(os.path.join(ruta_base, "backtest_df.parquet"))

# Renombrar columnas en todos los dataframes
train_df_ren = train_df.rename(columns={'semana': 'ds', 'y_usuarios_nuevos_semana': 'y'})
test_df_ren = test_df.rename(columns={'semana': 'ds', 'y_usuarios_nuevos_semana': 'y'})
backtest_df_ren = backtest_df.rename(columns={'semana': 'ds', 'y_usuarios_nuevos_semana': 'y'})

# Separar X (fechas) e y (objetivo) para cada conjunto
X_train = train_df_ren[['ds']]
y_train = train_df_ren['y']

X_test = test_df_ren[['ds']]
y_test = test_df_ren['y']

X_backtest = backtest_df_ren[['ds']]
y_backtest = backtest_df_ren['y']

## AutoML Use

In [0]:
# Horizonte de pronóstico: 1 meses
horizon = 24
# --- ENTRENAMIENTO -------------------------------------------
log_path = "/Workspace/Users/jorgee.lopez@adres.gov.co/mlops_canvas/notebooks/data_prueba/training_log.txt"
automl = AutoML()
automl.fit(
    X_train=X_train,
    y_train=y_train,
    period=horizon,
    task="ts_forecast",
    time_budget=360,
    metric="mape",
    log_file_name=log_path,
    eval_method="holdout"
)


In [0]:
# --- PRONÓSTICO: TEST Y BACKTEST ------------------------------

# Última fecha del entrenamiento
last_date = train_df_ren['ds'].max()

# Crear fechas futuras para los 24 pasos completos
X_future = pd.DataFrame({
    'ds': pd.date_range(start=last_date + pd.offsets.MonthBegin(1), periods=24, freq='MS')
})

# Predicción completa
y_pred_future = automl.predict(X_future)

# Dividir en test (primeros 16) y backtest (últimos 8)
y_pred_test = y_pred_future[:16]
y_pred_back = y_pred_future[16:]
X_test = X_future[:16]
X_backtest = X_future[16:]

In [0]:
# Unir test y backtest (valores reales y fechas)
X_real_total = pd.concat([X_test, X_backtest]).reset_index(drop=True)
y_real_total = pd.concat([y_test, y_backtest]).reset_index(drop=True)

# Unir predicciones (ya que fueron generadas con 24 fechas futuras)
X_pred_total = pd.concat([X_test, X_backtest]).reset_index(drop=True)
y_pred_total = pd.concat([y_pred_test, y_pred_back]).reset_index(drop=True)

# Graficar
plt.figure(figsize=(12, 6))

# Línea azul: valores reales (Test + Backtest)
plt.plot(X_real_total['ds'], y_real_total, marker='o', linestyle='-', color='blue', label='Real (Test + Backtest)')

# Línea naranja: predicción en test (primeros 16)
plt.plot(X_test['ds'], y_pred_test, marker='x', linestyle='--', color='orange', label='Predicción (Test)')

# Línea verde: predicción en backtest (últimos 8)
plt.plot(X_backtest['ds'], y_pred_back, marker='x', linestyle='--', color='green', label='Predicción (Backtest)')

# Estética
plt.xlabel("Fecha")
plt.ylabel("Usuarios únicos")
plt.title("Pronóstico: Valores reales y predicciones (24 meses)")
plt.grid(True)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
# Unir los datos reales (train + test + backtest)
X_real_full = pd.concat([train_df_ren[['ds']], X_test, X_backtest]).reset_index(drop=True)
y_real_full = pd.concat([train_df_ren['y'], y_test, y_backtest]).reset_index(drop=True)

# Unir las predicciones (ya generadas previamente)
X_pred_total = pd.concat([X_test, X_backtest]).reset_index(drop=True)
y_pred_total = pd.concat([y_pred_test, y_pred_back]).reset_index(drop=True)

# Crear la figura
plt.figure(figsize=(14, 6))

#  Línea azul: todos los valores reales históricos
plt.plot(X_real_full['ds'], y_real_full, marker='o', linestyle='-', color='blue', label='Real (Histórico + Test + Backtest)')

#  Línea naranja: predicciones en test
plt.plot(X_test['ds'], y_pred_test, marker='x', linestyle='--', color='orange', label='Predicción (Test)')

#  Línea verde: predicciones en backtest
plt.plot(X_backtest['ds'], y_pred_back, marker='x', linestyle='--', color='green', label='Predicción (Backtest)')

# Estética
plt.xlabel("Fecha")
plt.ylabel("Usuarios únicos")
plt.title("Evolución de usuarios únicos + Pronósticos (Test y Backtest)")
plt.grid(True)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    max_error
)
# --- MAPE personalizado -----------------------------------------
def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# --- FUNCIÓN DE MÉTRICAS ----------------------------------------
def calcular_metricas(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "MAPE (%)": mape(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "MedAE": median_absolute_error(y_true, y_pred),
        "Max Error": max_error(y_true, y_pred)
    }

# --- APLICAR MÉTRICAS A TEST Y BACKTEST -------------------------
metricas_test = calcular_metricas(y_test, y_pred_test)
metricas_back = calcular_metricas(y_backtest, y_pred_back)

# --- FORMAR TABLA -----------------------------------------------
df_metrics = pd.DataFrame([metricas_test, metricas_back], index=["Test", "Backtest"])
print(df_metrics.round(4).to_string())
